# 04 — Goal 3: multidimensional structure and reference robustness

Evaluates family organization, pairwise support, segmentation/encoding sensitivity, and the exact-session Rest reference.

Every displayed denominator and paper-facing visual is also saved under `outputs/visualization/`. Empty or under-supported analyses remain visible as audit rows; they are never silently removed.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from inside paper1_pipeline_rebuilt.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
VIZ_ROOT = OUTPUT / "visualization"
sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def read_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing required stage table: {parquet} or {csv}")

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def save_table(frame, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    path = target / f"{name}.csv"
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    png = target / f"{name}.png"
    svg = target / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("Visualization outputs:", VIZ_ROOT)


In [ ]:
RUN_GOAL_3_SENSITIVITY = False
if RUN_GOAL_3_SENSITIVITY:
    run_cli("extract", "--profile", "conservative")
    run_cli("extract", "--profile", "permissive")
    run_cli("sensitivity")
    run_cli("encoding-sensitivity")
    run_cli("rest-reference")

from paper1_qc.registry import metric_registry_frame
registry = metric_registry_frame()
pairwise = read_stage("04_analysis/descriptive/pairwise_clustered_spearman")


In [ ]:
features = registry.loc[
    registry["feature"].isin(set(pairwise["feature_left"]) | set(pairwise["feature_right"])),
    "feature",
].tolist()
rho = pd.DataFrame(np.eye(len(features)), index=features, columns=features)
n_participants = pd.DataFrame(np.nan, index=features, columns=features)
for row in pairwise.itertuples():
    rho.loc[row.feature_left, row.feature_right] = row.rho
    rho.loc[row.feature_right, row.feature_left] = row.rho
    n_participants.loc[row.feature_left, row.feature_right] = row.n_participants
    n_participants.loc[row.feature_right, row.feature_left] = row.n_participants

family_order = (
    registry.set_index("feature").loc[features, "family"].sort_values(kind="stable").index.tolist()
)
rho = rho.loc[family_order, family_order]
n_participants = n_participants.loc[family_order, family_order]
save_table(rho.reset_index(names="feature"), "04_goal3", "spearman_matrix")
save_table(n_participants.reset_index(names="feature"), "04_goal3", "pairwise_participant_support")

fig, axes = plt.subplots(1, 2, figsize=(19, 8))
sns.heatmap(rho, vmin=-1, vmax=1, center=0, cmap="vlag", ax=axes[0], square=True)
axes[0].set_title("Pairwise Spearman structure")
sns.heatmap(n_participants, cmap="viridis", ax=axes[1], square=True)
axes[1].set_title("Pair-specific participant denominator")
for ax in axes:
    ax.tick_params(axis="x", labelrotation=90, labelsize=7)
    ax.tick_params(axis="y", labelsize=7)
fig.tight_layout()
save_figure(fig, "04_goal3", "correlation_and_support_matrices")
plt.show()


In [ ]:
# Family coherence statistic with a fixed-label permutation null.
family_lookup = registry.set_index("feature")["family"].to_dict()
observed_pairs = pairwise.loc[pairwise["rho"].notna()].copy()
observed_pairs["same_family"] = (
    observed_pairs["feature_left"].map(family_lookup)
    == observed_pairs["feature_right"].map(family_lookup)
)
observed_stat = (
    observed_pairs.loc[observed_pairs["same_family"], "rho"].abs().mean()
    - observed_pairs.loc[~observed_pairs["same_family"], "rho"].abs().mean()
)

rng = np.random.default_rng(20260713)
feature_labels = pd.Series(family_lookup)
null = []
for _ in range(5000):
    shuffled = pd.Series(rng.permutation(feature_labels.values), index=feature_labels.index)
    same = (
        observed_pairs["feature_left"].map(shuffled)
        == observed_pairs["feature_right"].map(shuffled)
    )
    if same.any() and (~same).any():
        null.append(
            observed_pairs.loc[same, "rho"].abs().mean()
            - observed_pairs.loc[~same, "rho"].abs().mean()
        )
null = np.asarray(null)
p_value = (1 + np.sum(null >= observed_stat)) / (1 + len(null))
coherence = pd.DataFrame([{
    "observed_within_minus_between_abs_rho": observed_stat,
    "permutations": len(null),
    "one_sided_permutation_p": p_value,
    "seed": 20260713,
}])
save_table(coherence, "04_goal3", "family_coherence_permutation")
display(coherence)

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(null, bins=50, ax=ax, color="0.55")
ax.axvline(observed_stat, color="#C44E52", linewidth=2, label="Observed")
ax.set(title="Permutation null for family coherence", xlabel="Within-family |rho| − between-family |rho|")
ax.legend(frameon=False)
save_figure(fig, "04_goal3", "family_coherence_permutation")
plt.show()


In [ ]:
# Segmentation and native-encoding robustness are sensitivities, not new observations.
segmentation_robustness = read_stage("04_analysis/sensitivity/segmentation_profile_robustness")
save_table(segmentation_robustness, "04_goal3", "segmentation_profile_robustness")
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=segmentation_robustness, x="profile", y="spearman_rho", ax=ax)
sns.stripplot(data=segmentation_robustness, x="profile", y="spearman_rho", color="0.25", alpha=.6, ax=ax)
ax.set(title="Q-metric rank robustness to segmentation profiles", ylabel="Paired Spearman rho")
save_figure(fig, "04_goal3", "segmentation_profile_metric_robustness")
plt.show()

encoding = read_stage("04_analysis/encoding_sensitivity/paired_encoding_robustness")
save_table(encoding, "04_goal3", "paired_wav_webm_robustness")
display(encoding.sort_values("spearman_rho").head(20))


In [ ]:
# Rest is a matched acquisition-context sensitivity, never a speech-VAD input.
rest = read_stage("04_analysis/rest_reference/exact_session_bamboo_rest_comparison")
rest_summary = pd.DataFrame([{
    "exact_session_pairs": rest["logical_recording_id_bamboo"].nunique(),
    "participants": rest["SubjectID"].nunique(),
    "complete_level_pairs": rest[[
        "qadd_nonspeech_level_dbfs", "restref_level_dbfs"
    ]].dropna().shape[0],
}])
save_table(rest_summary, "04_goal3", "rest_exact_session_support")
display(rest_summary)

complete = rest[["SubjectID", "qadd_nonspeech_level_dbfs", "restref_level_dbfs"]].dropna().copy()
if len(complete):
    rho_rest = complete[["qadd_nonspeech_level_dbfs", "restref_level_dbfs"]].corr(method="spearman").iloc[0, 1]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.scatterplot(data=complete, x="restref_level_dbfs", y="qadd_nonspeech_level_dbfs", ax=axes[0])
    axes[0].set(title=f"Exact-session Rest vs Bamboo pause level (rho={rho_rest:.2f})")
    mean_level = complete[["qadd_nonspeech_level_dbfs", "restref_level_dbfs"]].mean(axis=1)
    difference = complete["qadd_nonspeech_level_dbfs"] - complete["restref_level_dbfs"]
    axes[1].scatter(mean_level, difference, alpha=.7)
    axes[1].axhline(difference.mean(), color="0.2")
    axes[1].axhline(difference.mean() + 1.96*difference.std(ddof=1), color="0.5", linestyle="--")
    axes[1].axhline(difference.mean() - 1.96*difference.std(ddof=1), color="0.5", linestyle="--")
    axes[1].set(title="Contextual agreement view", xlabel="Mean level (dBFS)", ylabel="Bamboo pause − Rest (dB)")
    fig.tight_layout()
    save_figure(fig, "04_goal3", "rest_reference_level_comparison")
    plt.show()
